# DeepFreq: proposed improvements (summary)

**Original DeepFreq** has two networks:
- **Network A (finds the frequencies):** 1 input layer, 20 conv layers with 64 filters each, and 1 upsampling output layer, about **1,050,000 parameters**.
- **Network B (counts them):** 20 conv layers with 32 filters each, giving one number (a regression), about **64,000 parameters**.

## Improvement 1: add a clean-up step after network A
- Layers / parameters / architecture: **no change**. It isn't a neural network.
- What it is: a maths step (least-squares fitting, 10 iterations) that moves each peak to its exact position.
- Result: frequencies are **3–7× more precise** at 20 dB and above.

## Improvement 2: replace network B with a new counter
- Architecture: 20 conv layers → **3 fully connected layers (an MLP)**.
- Layers: 20 → **3** (41 → 128 → 128 → 10).
- Parameters: 64,000 → **23,000** (about 3× fewer).
- Output: one number, rounded (regression) → **choose 1 of 10 counts** (classification).
- Input: network A's output only → network A's 15 highest peaks **plus** 25 classical signal features (eigenvalues) **plus** the old counter's guess.
- Result: correct counts go **up 5–11 points** (e.g. 76% → 84% at 20 dB).

## Improvement 3: try different designs for network A (tested, but not adopted)

| Version | Layers | Filters | Parameters | vs original |
| --- | --- | --- | --- | --- |
| Original | 20 conv | 64 | 1,050,000 | (baseline) |
| Small | 8 conv | 32 | 426,000 | **7× faster**, slightly less accurate |
| Residual (ResFreq) | 20 conv, with skip connections | 64 | 1,050,000 | less accurate |
| Attention | 2 transformer | 64 | 877,000 | not fully trained |
| Dilated | 8 conv (4 blocks) | 32 | 426,000 | least accurate |

So network A itself was **not** improved: the original design was still the most accurate.

## Final proposed system

Original network A (unchanged) → new counter (Improvement 2) → clean-up step (Improvement 1). Overall error drops **31% at 20 dB and 58% at 50 dB**. If speed matters more, swap in the small network A.